# Dataset Generation

Generates a synthetic clothing color dataset using `DatasetGenerator`.

Each sample is a 224×224 image with a colored inner square (pattern + fold texture + global lighting augmentation) composited onto a real indoor background.

**Output format**
- `dataset/images/` — JPEG images named `{split}_{index:06d}.jpg`
- `dataset/labels.csv` — one row per image: `filename, red, orange, ..., olive` where values are in `[0, 1]` and sum to `1.0`

**Task framing:** Soft multi-class classification. The label vector is a probability distribution over 13 color categories, making it suitable for a softmax head trained with KL-divergence or soft cross-entropy loss.

---
*For dataset preparation (background cleaning, color library), see [dataset_preparation.ipynb](dataset_preparation.ipynb).*

In [1]:
import os, sys, importlib, csv, random
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

root_path = os.path.abspath('..')
if root_path not in sys.path:
    sys.path.append(root_path)

import utils.dataset_utils as dataset_utils
importlib.reload(dataset_utils)

# ── Configuration ────────────────────────────────────────────────────────────
BG_DIR        = "../datasets/indoorCVPR_09_modified"
OUTPUT_DIR    = "../datasets/generated"
N_TRAIN       = 20_000
N_VAL         = 2_000
SEED          = 42

# All 13 categories in a fixed canonical order
COLOR_CLASSES = [
    "red", "orange", "yellow", "green", "blue",
    "violet", "purple", "white", "gray", "black",
    "pink", "brown", "olive",
]

random.seed(SEED)
np.random.seed(SEED)

img_dir = os.path.join(OUTPUT_DIR, "images")
os.makedirs(img_dir, exist_ok=True)
print(f"Output: {os.path.abspath(OUTPUT_DIR)}")
print(f"Splits: train={N_TRAIN}, val={N_VAL}  |  total={N_TRAIN + N_VAL}")

Output: c:\Users\lifei\OneDrive\Desktop\Personal Projects\BBAIRL\Clothing-Color-Recognition-ML-With-Synthetic-Dataset\datasets\generated
Splits: train=20000, val=2000  |  total=22000


## Generation Loop

In [ ]:
dg = dataset_utils.DatasetGenerator(path_to_bgs=BG_DIR)

csv_path = os.path.join(OUTPUT_DIR, "labels.csv")
splits = [("train", N_TRAIN), ("val", N_VAL)]

# ── Resume support ────────────────────────────────────────────────────────────
# Build the set of filenames already written so we can skip them on re-runs.
already_done = set()
csv_exists = os.path.isfile(csv_path)
if csv_exists:
    import pandas as pd
    existing = pd.read_csv(csv_path, usecols=["filename"])
    already_done = set(existing["filename"].tolist())
    print(f"Resuming — {len(already_done)} images already generated.")
else:
    print("Starting fresh.")

write_mode = "a" if csv_exists else "w"

with open(csv_path, write_mode, newline="") as f:
    writer = csv.writer(f)
    if not csv_exists:
        writer.writerow(["filename", "split"] + COLOR_CLASSES)

    for split_name, n in splits:
        for i in tqdm(range(n), desc=split_name):
            filename = f"{split_name}_{i:06d}.jpg"
            if filename in already_done:
                continue

            img_bgr, label_pcts = dg.generate()

            # Build normalized label vector (sums to 1.0)
            total = sum(label_pcts.values()) or 1.0
            label_vec = [round(label_pcts.get(c, 0.0) / total, 6) for c in COLOR_CLASSES]

            img_path = os.path.join(img_dir, filename)
            cv2.imwrite(img_path, img_bgr, [cv2.IMWRITE_JPEG_QUALITY, 95])
            writer.writerow([filename, split_name] + label_vec)

print(f"\nDone. CSV saved to: {csv_path}")

ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

## Verification

In [ ]:
import pandas as pd

df = pd.read_csv(csv_path)
print(f"Total rows : {len(df)}")
print(f"Train      : {(df['split'] == 'train').sum()}")
print(f"Val        : {(df['split'] == 'val').sum()}")
print(f"\nLabel sum check (should all be ~1.0):")
label_sums = df[COLOR_CLASSES].sum(axis=1)
print(f"  min={label_sums.min():.4f}  max={label_sums.max():.4f}  mean={label_sums.mean():.4f}")
print(f"\nMean label distribution across dataset:")
print(df[COLOR_CLASSES].mean().sort_values(ascending=False).to_string())

In [ ]:
# Visual spot-check — 5 random samples with their label distributions
sample_rows = df[df["split"] == "train"].sample(5, random_state=0)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for col, (_, row) in enumerate(sample_rows.iterrows()):
    img = cv2.imread(os.path.join(img_dir, row["filename"]))
    axes[0, col].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0, col].axis("off")
    axes[0, col].set_title(row["filename"], fontsize=7)

    present = {c: row[c] for c in COLOR_CLASSES if row[c] > 0.01}
    axes[1, col].barh(list(present.keys()), list(present.values()), color="steelblue")
    axes[1, col].set_xlim(0, 1)
    axes[1, col].tick_params(labelsize=7)
    axes[1, col].set_xlabel("weight", fontsize=7)

plt.tight_layout()
plt.show()